In [3]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
import keras

from keras.src.legacy.preprocessing.image import ImageDataGenerator
from keras.src import applications
from keras.models import Sequential, load_model
from keras.src.layers import Conv2D, MaxPooling2D, GlobalAveragePooling2D, Flatten, Dense, Dropout
from keras.src.legacy.preprocessing import image

import cv2
import os

import warnings
warnings.filterwarnings('ignore')

2026-09-15 06:39:39.357090: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-09-15 06:39:39.357148: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-09-15 06:39:39.358559: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [4]:
# Data Augmentation on train dataset
train_datagen = ImageDataGenerator(
    rescale=1./255,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True
)

# Data Augmentation on test dataset
test_datagen = ImageDataGenerator(
    rescale=1./255
)

In [5]:
cd ..

/kaggle


In [6]:
cd ..

/


In [7]:
train_generator = train_datagen.flow_from_directory(
    'kaggle/input/data/train',
    target_size=(255,255),
    batch_size=32,
    class_mode='categorical'
)

valid_generator = test_datagen.flow_from_directory(
    'kaggle/input/data/valid',
    target_size=(255,255),
    batch_size=32,
    class_mode='categorical'
)

Found 13104 images belonging to 15 classes.
Found 300 images belonging to 15 classes.


In [8]:
model = Sequential()
model.add(Conv2D(32,(3,3),input_shape=(255,255,3),activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Conv2D(64,(3,3),activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Flatten())
model.add(Dense(64,activation='relu'))
model.add(Dense(15,activation='softmax'))

model.compile(optimizer='adam',loss='categorical_crossentropy',metrics=['accuracy'])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 253, 253, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 126, 126, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 124, 124, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 62, 62, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 246016)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │    15,745,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 15)             │           975 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 15,765,455 (60.14 MB)

 Trainable params: 15,765,455 (60.14 MB)

 Non-trainable params: 0 (0.00 B)

In [9]:
history = model.fit(
        train_generator,
        batch_size=32,
        epochs=10,
        validation_data=valid_generator,
        validation_batch_size=32,
)

Epoch 1/10
  1/410 ━━━━━━━━━━━━━━━━━━━━ 2:03:09 18s/step - accuracy: 0.0312 - loss: 2.7215

I0000 00:00:1789454420.085286     204 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
W0000 00:00:1789454420.100489     204 graph_launch.cc:671] Fallback to op-by-op mode because memset node breaks graph update


409/410 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.2316 - loss: 2.8220

W0000 00:00:1789454925.986418     204 graph_launch.cc:671] Fallback to op-by-op mode because memset node breaks graph update


410/410 ━━━━━━━━━━━━━━━━━━━━ 530s 1s/step - accuracy: 0.2321 - loss: 2.8190 - val_accuracy: 0.3033 - val_loss: 2.4537
Epoch 2/10
410/410 ━━━━━━━━━━━━━━━━━━━━ 336s 804ms/step - accuracy: 0.4842 - loss: 1.6774 - val_accuracy: 0.3667 - val_loss: 2.2892
Epoch 3/10
410/410 ━━━━━━━━━━━━━━━━━━━━ 336s 805ms/step - accuracy: 0.5295 - loss: 1.4961 - val_accuracy: 0.4200 - val_loss: 2.2932
Epoch 4/10
410/410 ━━━━━━━━━━━━━━━━━━━━ 332s 796ms/step - accuracy: 0.5743 - loss: 1.3505 - val_accuracy: 0.3833 - val_loss: 2.2174
Epoch 5/10
410/410 ━━━━━━━━━━━━━━━━━━━━ 337s 808ms/step - accuracy: 0.6111 - loss: 1.2199 - val_accuracy: 0.4333 - val_loss: 2.2444
Epoch 6/10
410/410 ━━━━━━━━━━━━━━━━━━━━ 341s 815ms/step - accuracy: 0.6404 - loss: 1.1122 - val_accuracy: 0.4667 - val_loss: 2.0795
Epoch 7/10
410/410 ━━━━━━━━━━━━━━━━━━━━ 341s 817ms/step - accuracy: 0.6644 - loss: 1.0199 - val_accuracy: 0.5167 - val_loss: 2.0246
Epoch 8/10
410/410 ━━━━━━━━━━━━━━━━━━━━ 337s 809ms/step - accuracy: 0.6735 - loss: 0.9988 

In [10]:
model.save('WheatDiseaseDetection.h5')